### Plot the zotero collection hierarchy
Started from [here](https://www.perplexity.ai/search/using-pyzotero-find-all-subcat-9GGkb6t9TBKv5hsl.J5FSg)

In [ ]:
from pyzotero import zotero
import plotly.graph_objects as go
import plotly.graph_objects as go
import refwrangle as rfw

zot = zotero.Zotero(rfw.library_id, rfw.library_type, rfw.api_key)

# Fetch all collections
all_collections = zot.all_collections()

# Create a dictionary to store the hierarchy
hierarchy = {}

# Build the hierarchy
for collection in all_collections:
    key = collection['key']
    name = collection['data']['name']
    parent = collection['data'].get('parentCollection', '')
    hierarchy[key] = {'name': name, 'parent': parent, 'children': []}

# Populate children
for key, data in hierarchy.items():
    if data['parent']:
        hierarchy[data['parent']]['children'].append(key)

# Create a recursive function to build the tree
def build_tree(key):
    node = hierarchy[key]
    children = [build_tree(child) for child in node['children']]
    return {'name': node['name'], 'children': children}

# Find the root collections
root_collections = [key for key, data in hierarchy.items() if not data['parent']]

# Build the tree structure
tree_data = [build_tree(root) for root in root_collections]

# Create lists to hold the labels and parents
labels = []
parents = []

# Function to flatten the tree structure
def flatten_tree(node, parent=""):
    labels.append(node['name'])
    parents.append(parent)
    for child in node.get('children', []):
        flatten_tree(child, node['name'])

# Flatten the tree structure
for root in tree_data:
    flatten_tree(root)

# Create the treemap
fig = go.Figure(go.Treemap(
    labels=labels,
    parents=parents,
    root_color="lightgrey"
))

# Update the layout
fig.update_layout(
    title="Zotero Collection Hierarchy",
    width=1000,
    height=800
)

# Show the plot
fig.show()
